# What the FPGA buys you for microGPT

This notebook demonstrates three concrete wins from running the TALOS-V2 microGPT core on the PYNQ-Z2 PL rather than as a Python program on the PS:

1. **Tightly bounded latency.** The FPGA reports cycle-accurate timing for every generation through `perf_cycles_reg`. Run-to-run jitter is exactly zero for a fixed seed and within a handful of cycles for a varying seed — no GC pauses, no Python interpreter overhead.
2. **Bit-exact determinism.** Same seed → same tokens, byte for byte, every time. The categorical sampler is reproducible because the LFSR-fed RNG is deterministic in silicon.
3. **Empirical sampling distribution at zero marginal cost.** Because each generation is a fast register transaction (a few thousand FPGA cycles), we can sweep thousands of seeds in seconds and recover the model's *empirical* token distribution at each position — something that would take orders of magnitude longer in a software-only tiny-model implementation.

Each section below is small enough that you can execute it in a few seconds, and each one ends with a concrete number.

## 0. Setup

We point at the driver that lives next to this notebook tree (`../drivers/microgpt.py`), construct the `MicroGPT` host object, and program the bitstream. After this cell, the PL is alive and `gpt.status()` reports `ready=True`.

In [ ]:
import sys, time, statistics
from collections import Counter
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

drivers_path = (Path('..') / 'drivers').resolve()
if str(drivers_path) not in sys.path:
    sys.path.insert(0, str(drivers_path))

from microgpt import MicroGPT, TOKEN_ALPHABET, BOS_TOKEN_ID

gpt = MicroGPT()
print('overlay loaded; status =', gpt.status())
FCLK_HZ = 50_000_000  # PS-driven FCLK_CLK0
VOCAB = list(TOKEN_ALPHABET) + ['<bos>']  # 26 letters + sentinel

## 1. Deterministic latency (the headline number)

We run `generate()` 50 times with the same parameters, read the FPGA's `perf_cycles` counter for each run, and convert to wall-clock latency at 50 MHz.

**What we expect:** every run produces *exactly* the same cycle count (because the seed and config are identical, the trellis through the FSM and core is identical), giving a measured *standard deviation of zero cycles*. That is impossible for a Python forward pass; even a JIT-compiled software inference path would show µs-scale jitter from GC, scheduler, and cache effects.

In [ ]:
N_RUNS = 50
MAX_TOK = 8
TEMP    = 1.0
SEED    = 42

cycle_samples = []
wall_samples  = []
for _ in range(N_RUNS):
    t0 = time.perf_counter()
    text, info = gpt.generate(max_tokens=MAX_TOK, temperature=TEMP, seed=SEED)
    t1 = time.perf_counter()
    cycle_samples.append(info['cycles'])
    wall_samples.append(t1 - t0)

cyc_mean = statistics.mean(cycle_samples)
cyc_std  = statistics.pstdev(cycle_samples)
ns_per_token = (cyc_mean / len(info['tokens'])) * (1e9 / FCLK_HZ)
wall_mean_ms = 1e3 * statistics.mean(wall_samples)
wall_std_ms  = 1e3 * statistics.pstdev(wall_samples)

print(f"runs                : {N_RUNS}")
print(f"text per run        : {text!r}")
print(f"FPGA cycles / run   : mean={cyc_mean:.1f}  stdev={cyc_std:.3f}")
print(f"FPGA cycles / token : {cyc_mean / len(info['tokens']):.1f}")
print(f"hardware latency    : {ns_per_token/1e3:.2f} us/token  (= {ns_per_token/1e6:.3f} ms/token at {FCLK_HZ/1e6:.0f} MHz)")
print(f"wall-clock per run  : {wall_mean_ms:.2f} +- {wall_std_ms:.2f} ms (PS overhead, AXI transactions)")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 3.5))
axs[0].hist(cycle_samples, bins=10, color='C0', edgecolor='k')
axs[0].set_xlabel('FPGA cycles per run'); axs[0].set_ylabel('count'); axs[0].set_title(f'cycle count, N={N_RUNS}')
axs[0].axvline(cyc_mean, color='C3', linestyle='--', label=f'mean = {cyc_mean:.0f}')
axs[0].legend()
axs[1].hist([w*1e3 for w in wall_samples], bins=15, color='C2', edgecolor='k')
axs[1].set_xlabel('wall-clock per run (ms)'); axs[1].set_ylabel('count'); axs[1].set_title('PS-side wall clock')
plt.tight_layout(); plt.show()

**Reading the chart.** The left histogram is a single bar at the exact cycle count: the hardware is bit-exact for the same seed. The right histogram is the wall-clock per run as observed by Python; the spread there is *entirely* host-side overhead (AXI transactions, status polling, the Python interpreter), and it represents the *floor* you would have to get above with any software-only reimplementation.

## 2. Bit-exact reproducibility from the seed

Run 5 generations with the *same* seed and check the output is byte-for-byte identical. Then run 8 generations with *different* seeds to confirm we get a varied set. Determinism for a fixed seed is what makes the hardware suitable for regression testing and reproducible benchmarks; diversity across seeds is what makes the categorical sampler useful for actual generation.

In [ ]:
out_same = [gpt.generate(max_tokens=8, temperature=1.0, seed=0xC0FFEE)[0] for _ in range(5)]
print('5 runs at seed=0xC0FFEE :', out_same)
assert all(o == out_same[0] for o in out_same), 'reproducibility check failed'

seeds = [1, 42, 1337, 0xDEADBEEF, 0xC0FFEE, 0xCAFEBABE, 7, 97]
for s in seeds:
    text, info = gpt.generate(max_tokens=8, temperature=1.0, seed=s)
    print(f"seed=0x{s:08X}  text={text!r:12s}  cycles={info['cycles']}")

## 3. Empirical token distribution by position

Because each generation is just a few thousand FPGA cycles, we can run the core with thousands of random seeds and reconstruct the *empirical* sampling distribution of the model — i.e. how often each letter appears at position 0, position 1, … . This is the kind of measurement that's expensive to do with a slow software inference path; here it takes a few seconds for a thousand generations.

It's also a real cross-check on the model: position 0 is always conditioned on BOS (so the distribution there should match what the model thinks BOS most often produces); later positions are conditioned on whatever was just sampled, so they spread out.

In [ ]:
N_SEEDS  = 1024
N_TOKENS = 8

rng = np.random.default_rng(2026)
seeds = rng.integers(1, 2**31 - 1, size=N_SEEDS, dtype=np.int64)

# pos_freq[pos][token_id] -> count
pos_freq = [Counter() for _ in range(N_TOKENS)]
total_cycles = 0
t0 = time.perf_counter()
for s in seeds:
    text, info = gpt.generate(max_tokens=N_TOKENS, temperature=1.0, seed=int(s))
    total_cycles += info['cycles']
    for pos, tok in enumerate(info['tokens'][:N_TOKENS]):
        pos_freq[pos][int(tok)] += 1
t_total = time.perf_counter() - t0
print(f'{N_SEEDS} generations of {N_TOKENS} tokens in {t_total:.2f} s')
print(f'total FPGA cycles : {total_cycles}  (= {total_cycles*1e3/(N_SEEDS*FCLK_HZ):.2f} ms of pure inference)')

In [ ]:
# Plot the per-position empirical distribution as a heatmap.
freq_matrix = np.zeros((N_TOKENS, len(VOCAB)))
for pos in range(N_TOKENS):
    total = sum(pos_freq[pos].values())
    if total == 0:
        continue
    for tok, c in pos_freq[pos].items():
        if 0 <= tok < len(VOCAB):
            freq_matrix[pos, tok] = c / total

fig, ax = plt.subplots(figsize=(11, 4))
im = ax.imshow(freq_matrix, aspect='auto', cmap='viridis', origin='lower')
ax.set_xticks(range(len(VOCAB)))
ax.set_xticklabels(VOCAB, fontsize=9)
ax.set_yticks(range(N_TOKENS))
ax.set_yticklabels([f'pos {i}' for i in range(N_TOKENS)])
ax.set_xlabel('sampled token')
ax.set_title(f'Empirical sampling distribution per position ({N_SEEDS} seeds, T=1.0)')
fig.colorbar(im, ax=ax, label='probability')
plt.tight_layout(); plt.show()

**Reading the heatmap.** Brighter cells are more likely outputs at that (position, token). Position 0 is always conditioned on BOS so its row reflects the model's prior on the first letter; later rows broaden because the model conditions on a different sample each run. If a single letter dominates a row, that's where the model is confident; uniform-looking rows mean the model is more uncertain and the temperature has more leverage.

What this section is really showing is throughput: 1024 generations × 8 tokens each ≈ 8 192 tokens, sampled in a few seconds *including* every Python register write/read. The on-chip core itself accounted for only the small fraction reported as "pure inference" above.

## Summary

* **Deterministic latency.** Every run with the same seed reports the same FPGA cycle count to the bit. The wall-clock spread you see is host overhead, not hardware non-determinism.
* **Repeatable sampling.** The on-chip RNG is fed by a 32-bit LFSR seeded over AXI, so the categorical sampler is bit-exact reproducible — `generate(..., seed=s)` is a function in the mathematical sense, not a Monte Carlo flavour of one. That property is invaluable for tests and for reproducing reported metrics.
* **Cheap empirical statistics.** A thousand generations is a few seconds of work, so building the per-position distribution heatmap is essentially free. A pure-Python inference of the same model would be wall-clock-bound long before we collect that many samples.

Once the bitstream is loaded, the PS only does AXI register writes/reads. There is no model state in DRAM, no weight movement per token, no Python in the critical path. That's the win.